# 🤖 AetherSwarm v1 — Google Colab (GPU T4)

Lance ton OS d'agents IA en local sur Colab avec GPU T4.  
**Temps de setup : ~5 minutes**

### Prérequis
- Runtime : **T4 GPU** (Exécution → Modifier le type d'exécution → T4 GPU)
- Ton fichier `AetherSwarm-v1.zip` (reçu après achat)
- Un compte ngrok gratuit → [ngrok.com](https://ngrok.com) pour récupérer ton authtoken

---

## Étape 1 — Vérifier le GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

## Étape 2 — Installer Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Ollama installé")

## Étape 3 — Démarrer Ollama et télécharger le modèle

In [ ]:
import subprocess, time, os

# Démarrer Ollama en arrière-plan
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0'
proc = subprocess.Popen(
    ['ollama', 'serve'],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)
print("✅ Ollama démarré")

# Télécharger llama3.2:3b (rapide, ~2GB)
print("⏳ Téléchargement llama3.2:3b...")
!ollama pull llama3.2:3b
print("✅ Modèle prêt")

## Étape 4 — Uploader AetherSwarm-v1.zip

In [ ]:
from google.colab import files
import zipfile, os

print("📁 Sélectionne ton fichier AetherSwarm-v1.zip")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"\n📦 Extraction de {zip_name}...")

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/aetherswarm')

# Trouver le dossier extrait
contents = os.listdir('/content/aetherswarm')
print(f"✅ Extrait : {contents}")

## Étape 5 — Installer les dépendances Python

In [ ]:
import os, glob

# Chercher requirements.txt
req_files = glob.glob('/content/aetherswarm/**/requirements*.txt', recursive=True)

if req_files:
    req_path = req_files[0]
    print(f"📋 requirements.txt trouvé : {req_path}")
    !pip install -q -r {req_path}
else:
    print("⚠️ requirements.txt non trouvé — installation des packages standard")
    !pip install -q fastapi uvicorn langchain langgraph langchain-community chromadb anthropic ollama httpx python-multipart

print("✅ Dépendances installées")

## Étape 6 — Configurer et lancer le backend AetherSwarm

In [ ]:
import glob, os

# Trouver main.py
main_files = glob.glob('/content/aetherswarm/**/main.py', recursive=True)
if not main_files:
    main_files = glob.glob('/content/aetherswarm/**/*.py', recursive=True)

print("Fichiers Python trouvés :")
for f in main_files[:10]:
    print(f"  {f}")

# Identifier le dossier backend
if main_files:
    backend_dir = os.path.dirname(main_files[0])
    print(f"\n📂 Dossier backend : {backend_dir}")
    os.environ['AETHER_DIR'] = backend_dir

In [ ]:
import subprocess, time, os

backend_dir = os.environ.get('AETHER_DIR', '/content/aetherswarm')

# Variables d'environnement
env = os.environ.copy()
env['OLLAMA_BASE_URL'] = 'http://localhost:11434'
env['LLM_MODEL'] = 'llama3.2:3b'
# env['ANTHROPIC_API_KEY'] = 'sk-ant-...'  # Décommenter si tu veux Claude API

# Lancer le backend FastAPI
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8787', '--reload'],
    cwd=backend_dir,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# Vérifier que le serveur tourne
import urllib.request
try:
    urllib.request.urlopen('http://localhost:8787')
    print("✅ Backend AetherSwarm démarré sur port 8787")
except Exception as e:
    print(f"⚠️ Vérification : {e}")
    print("(Normal si le backend retourne une page HTML)")
    print("✅ Backend probablement actif — continue")

## Étape 7 — Exposer avec ngrok

Récupère ton authtoken gratuit sur [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok, conf

# 🔑 Colle ton authtoken ngrok ici
NGROK_TOKEN = ""  # <-- colle ici ton token

if not NGROK_TOKEN:
    print("⚠️ Colle ton ngrok authtoken dans NGROK_TOKEN ci-dessus")
    print("Récupère-le sur : https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    conf.get_default().auth_token = NGROK_TOKEN
    tunnel = ngrok.connect(8787)
    url = tunnel.public_url
    print(f"""\n{'='*60}
🚀 AetherSwarm est accessible ici :

   {url}

Ouvre ce lien dans ton navigateur !
{'='*60}""")

## (Optionnel) Tester l'API directement

In [ ]:
import requests, json

# Test direct de l'API locale
try:
    r = requests.get('http://localhost:8787/api/health', timeout=5)
    print(f"Health check : {r.status_code}")
    print(r.json())
except Exception as e:
    print(f"Endpoint /api/health : {e}")
    print("Essaie /api/status ou /health selon la version d'AetherSwarm")

## (Optionnel) Lancer une mission via API

In [ ]:
import requests

mission = {
    "brief": "Analyse les tendances actuelles du marché des outils IA pour solopreneurs et propose 3 opportunités de produits à fort potentiel.",
    "mode": "swarm",
    "web_search": False,
    "memory": False
}

try:
    r = requests.post('http://localhost:8787/api/run', json=mission, timeout=120)
    print(f"Status : {r.status_code}")
    result = r.json()
    print(json.dumps(result, indent=2, ensure_ascii=False)[:3000])
except Exception as e:
    print(f"Erreur : {e}")
    print("Utilise l'interface graphique via le lien ngrok ci-dessus")

---
## Notes

- **Session Colab** : expire après ~12h (plan gratuit). Relance depuis Étape 3.
- **Mémoire Chroma** : non persistante entre sessions. Utilise Google Drive pour sauvegarder.
- **Modèle** : `llama3.2:3b` par défaut. Tu peux changer pour `qwen2.5:7b` ou `mistral:7b` si tu veux plus de puissance.
- **Claude API** : décommente `ANTHROPIC_API_KEY` à l'Étape 6 si tu as une clé Anthropic.

**Support** : mindset22633@gmail.com